# 09b · Istari v2 Semantic API vs Our System

Adds Istari **v2 semantic search** to the existing comparison.
Reuses all cached data from `result/09_api_evaluation/` — no need to re-run notebook 09.

| Method | Corpus | Type |
|--------|--------|------|
| Istari v1 BM25 | 20M GOI | Keyword search (cached) |
| **Istari v2 Semantic** | 20M GOI | Natural language semantic search (NEW) |
| Our MiniLM | 98K subsample | Dense embedding (cached) |

**Ground truth:** `goi_search_results.json` — graded similarity scores from Istari  
**Relevance threshold:** 0.75

## 1 · Environment Setup

In [ ]:
import os, json, time, pickle, requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

ISTARI_API_KEY   = os.getenv('API_KEY')
V2_BASE_URL      = 'https://api.istari.ai/v2/search'
V1_RESULT_DIR    = Path('result/09_api_evaluation')
RESULT_DIR       = Path('result/09b_api_v2')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANCE_THRESHOLD = 0.75
# K_VALUES is no longer hardcoded here — it's computed after the live quota
# check below, since the real per-query size depends on remaining budget.

print(f'[Setup] V2 URL        : {V2_BASE_URL}')
print(f'[Setup] API key set   : {"✅" if ISTARI_API_KEY else "❌  — check API_KEY in .env"}')
print(f'[Setup] Result dir    : {RESULT_DIR}/')

## 2 · v2 API Smoke Test

In [ ]:
print('[Setup] Testing v2 API...')
r = requests.post(
    V2_BASE_URL,
    headers={
        'Accept': 'application/json',
        'x-api-key': ISTARI_API_KEY,
        'Content-Type': 'application/json',
    },
    json={
        'describe': 'software companies',
        'keywords': {'must_all': [], 'must_any': [], 'must_not': []},
        'filters':  {'country': [], 'state': [], 'region': [],
                     'organization_type': [], 'organization_size': [], 'nace_code': []},
        'excludes': [],
        'columns':  ['domain', 'name'],
        'size':     3,
    },
    timeout=15,
)
if r.status_code == 200:
    sample = r.json().get('data', [])
    print(f'[Setup] v2 API OK ✅  — got {len(sample)} results')
    print(f'[Setup] Sample      : {sample[0].get("name","?")} ({sample[0].get("domain","?")})')
else:
    print(f'[Setup] v2 API ERROR: {r.status_code} {r.text[:200]}')
    raise SystemExit('Fix API key or endpoint before continuing.')

# Live quota check — the server reports the real remaining budget on every
# response via these headers. This is authoritative; don't trust dashboard
# UI fields like "Last Used" which can be stale/broken independent of the
# actual server-enforced counters.
#
# A MISSING header is treated as "unlimited", not zero. An unlimited-tier API
# key may simply not send these headers at all (nothing to rate-limit), and
# defaulting a missing header to 0 would silently self-limit an unlimited key
# down to a tiny SIZE_PER_QUERY instead of the full TARGET_SIZE_PER_QUERY. Raw
# header values are printed below so this is verifiable rather than assumed.
REMAINING_RESULTS_HDR  = r.headers.get('X-RateLimit-Results-Remaining')
REMAINING_REQUESTS_HDR = r.headers.get('X-RateLimit-Requests-Remaining')
UNLIMITED_FALLBACK     = 10_000_000

REMAINING_RESULTS  = int(REMAINING_RESULTS_HDR)  if REMAINING_RESULTS_HDR  is not None else UNLIMITED_FALLBACK
REMAINING_REQUESTS = int(REMAINING_REQUESTS_HDR) if REMAINING_REQUESTS_HDR is not None else UNLIMITED_FALLBACK

print(f'[Setup] Raw rate-limit headers  : results={REMAINING_RESULTS_HDR!r}, requests={REMAINING_REQUESTS_HDR!r}')
if REMAINING_RESULTS_HDR is None or REMAINING_REQUESTS_HDR is None:
    print('[Setup] NOTE: header(s) missing from response — treating as unlimited for this run '
          '(expected for an unlimited-quota API key). If this is wrong, the raw values above '
          'will show it.')
print(f'[Setup] Live quota remaining this month → requests: {REMAINING_REQUESTS}, results: {REMAINING_RESULTS}')

## 3 · Load Queries

In [ ]:
print('[Load] Loading queries...')
queries_df   = pd.read_excel('dataset/queries.xlsx')
query_col    = next(c for c in queries_df.columns if queries_df[c].dtype == object)
TEST_QUERIES = queries_df[query_col].astype(str).tolist()
print(f'[Load] {len(TEST_QUERIES)} queries from column "{query_col}"')
print(f'[Load] First 5: {TEST_QUERIES[:5]}')

## 4 · Load Cached v1 BM25 + MiniLM Results

These were computed in notebook `09_api_evaluation` and are reused as-is.

In [ ]:
print('[Load] Loading cached results from result/09_api_evaluation/...')

with open(V1_RESULT_DIR / 'api_results_cache.pkl', 'rb') as f:
    api_results, api_latencies = pickle.load(f)
print(f'[Load] v1 BM25 cached : {len(api_results["bm25"])} queries ✅')

with open(V1_RESULT_DIR / 'minilm_results_cache.pkl', 'rb') as f:
    our_results, our_latencies = pickle.load(f)
print(f'[Load] MiniLM cached  : {len(our_results)} queries ✅')

## 5 · Fetch v2 Semantic Results

v2 takes a natural language `describe` field. Fixes vs the original version of this notebook:

- **Real pagination via `search_after`.** The old code paginated using a `from` offset field that isn't part of the real v2 schema. Confirmed against Istari's `/api/goi-search` reference: the real mechanism is an opaque cursor — send `search_after` (omit for page 1), read the next cursor back from the response's `metadata.search_after` (`null` = no further pages). `size` maxes at 500/request, but scored/semantic search allows up to 10,000 rows of total depth, so fetching up to 1000/query (matching the `k=1000` used across the other baselines) takes at most 2 pages/query. Proof the *old* fake pagination was broken: its cached "200-result" entries only had **50 unique domains each** — it was silently re-fetching page 1 four times instead of actually paginating.
- **Two different 429s, handled differently.** A 429 with `X-RateLimit-Results-Remaining` near 0 (or a quota-specific message) means the *monthly* quota is genuinely exhausted — retrying never helps, so this is fatal and stops the run. A 429 with plenty of results still remaining (e.g. plain `{"message":"Too Many Requests"}`) is a *transient* per-second/minute throttle — confirmed live: firing 101 requests at 1/sec tripped this after just 10 queries, with 3,184/3,494 results still unused. That case gets a short retry (10s/20s/.../50s) instead of either ignoring it or aborting.
- **No cache poisoning.** A failed fetch is never written to the results cache, so a real error can't get silently treated as "zero relevant results" (which is what produced the all-zero NDCG/Recall rows in the original run).
- **Budget-aware size and cost, on both quota dimensions, based on actual remaining work.** Results quota and request quota are tracked separately by the API. `SIZE_PER_QUERY` is computed from the live *results* budget, capped at the `TARGET_SIZE_PER_QUERY` of 1000. The cache is loaded and filtered for staleness *before* estimating request cost, so the request-budget check reflects only the queries that still actually need fetching this run — not a worst-case "refetch all 101" estimate, which could otherwise needlessly abort a run that only needs to resume a couple of missing queries once the remaining request quota gets tighter later in the project.
- **Deduplicated, bounded pagination.** Near-tied ranking scores can legitimately cause the same domain to reappear at the page-1/page-2 boundary (observed live: 1-15 duplicates out of 1000, i.e. <1.5% — normal cursor-pagination behaviour, not the ~75%-duplication old bug). `v2_call()` dedupes by domain across pages and caps at `max_pages` requests, accepting a small bounded shortfall (e.g. 985/1000) rather than chasing the last few results with an open-ended number of extra requests.
- **Resumable, depth-aware cache with tolerance.** Cached entries are dropped (and refetched) if they're poisoned (empty, or fewer unique domains than results — the old pagination bug's signature) **or** meaningfully shallower than the current run's `SIZE_PER_QUERY` (below `SHALLOW_TOLERANCE` = 90% of it) — e.g. leftover `size=28` results from an earlier, smaller-quota run. A small, legitimate dedup-driven shortfall (e.g. 985/1000) is *not* treated as stale, so a run doesn't re-fetch (and re-spend quota on) roughly half its queries every time purely because of harmless page-boundary overlap.

In [ ]:
import math

QUERIES_TO_FETCH = len(TEST_QUERIES)  # 101 total queries in the evaluation set

SAFETY_MARGIN         = 0.9    # leave 10% headroom on top of what the server currently reports
V2_MAX_PAGE_SIZE       = 500   # documented API max per single request
TARGET_SIZE_PER_QUERY  = 1000  # desired depth -- matches k=1000 used across the other
                                # baselines; well under the documented 10,000-row depth
                                # cap for scored/semantic search modes

SIZE_PER_QUERY = min(
    TARGET_SIZE_PER_QUERY,
    int((REMAINING_RESULTS * SAFETY_MARGIN) // QUERIES_TO_FETCH),
)

if SIZE_PER_QUERY < 5:
    raise SystemExit(
        f'[v2] Only {REMAINING_RESULTS} results left this month — '
        f'not enough for {QUERIES_TO_FETCH} queries even at size=5. Stop and wait for quota reset.'
    )

PAGES_PER_QUERY = math.ceil(SIZE_PER_QUERY / V2_MAX_PAGE_SIZE)  # 1 or 2 pages to reach up to 1000

# ── Load cache and drop stale entries BEFORE estimating request cost ──────────
# Doing this here (rather than after a budget check based on all 101 queries)
# means the budget estimate below reflects only the queries that ACTUALLY still
# need fetching this run. Without this, a run that only needs to resume 1-2
# missing queries could be needlessly aborted just because a worst-case
# "refetch all 101" estimate exceeds the currently remaining request quota,
# even though the real (tiny) need would easily fit.
SHALLOW_TOLERANCE = 0.9  # a cached entry within 90% of SIZE_PER_QUERY is NOT considered
                          # stale -- dedup against page-boundary overlaps (see v2_call)
                          # can legitimately leave a query a few results short of the
                          # exact target, which must not trigger a needless refetch

v2_cache_path = RESULT_DIR / 'v2_results_cache.pkl'
if v2_cache_path.exists():
    with open(v2_cache_path, 'rb') as f:
        v2_results, v2_latencies = pickle.load(f)
    print(f'[v2] Loading cache from a previous run: {len(v2_results)}/{len(TEST_QUERIES)} queries')

    # Drop entries that need refetching rather than trusting them as-is:
    #   - poisoned by the ORIGINAL buggy version of this notebook: empty list (a past 429
    #     got cached as if it were a real zero-result response), or fewer unique domains
    #     than results returned (broken `from`-offset pagination silently duplicating page 1
    #     at a ~75% rate -- NOT the same as the small page-boundary overlap v2_call already
    #     dedupes against, which is why this check is now redundant for anything fetched by
    #     the current logic, but still catches any leftover pre-fix cache)
    #   - meaningfully shallower than the CURRENT quota-based SIZE_PER_QUERY (below the
    #     SHALLOW_TOLERANCE fraction of it): a cache built when less monthly quota (or a
    #     smaller TARGET_SIZE_PER_QUERY) was available must not be silently reused as "done"
    #     once more quota is available -- that would permanently cap this run's depth at
    #     whatever the smallest historical quota happened to allow.
    # A cache entry that is neither poisoned nor meaningfully shallow is left alone, so a
    # run can still resume after an interruption without re-fetching (and re-spending quota
    # on) queries already fetched at (or acceptably close to) the current target depth.
    stale = [
        q for q, data in v2_results.items()
        if not data
        or len({r['domain'] for r in data}) < len(data)
        or len(data) < SIZE_PER_QUERY * SHALLOW_TOLERANCE
    ]
    for q in stale:
        del v2_results[q]
        v2_latencies.pop(q, None)
    if stale:
        print(f'[v2] Dropped {len(stale)} poisoned/shallow cached entries (refetching at size={SIZE_PER_QUERY}): {stale}')
    print(f'[v2] Valid cached queries at current depth: {len(v2_results)}/{len(TEST_QUERIES)}')
else:
    v2_results   = {}
    v2_latencies = {}
    print('[v2] No cache — starting fresh')

to_fetch = [q for q in TEST_QUERIES if q not in v2_results]

# Real pagination (search_after cursor) means each query can cost more than
# one request now -- check the SEPARATE request-quota budget too, not just
# the results budget above. Based on the ACTUAL number of queries left to
# fetch this run, not a worst-case full-refetch assumption.
est_results  = SIZE_PER_QUERY * len(to_fetch)
est_requests = PAGES_PER_QUERY * len(to_fetch)

if est_requests > REMAINING_REQUESTS:
    raise SystemExit(
        f'[v2] Fetching the {len(to_fetch)} still-needed queries needs up to ~{est_requests} '
        f'requests ({PAGES_PER_QUERY}/query) but only {REMAINING_REQUESTS} requests remain '
        f'this month. Lower TARGET_SIZE_PER_QUERY or wait for quota reset.'
    )

K_VALUES = sorted({k for k in [10, 30, 50, 100, 200, 500] if k <= SIZE_PER_QUERY} | {SIZE_PER_QUERY})

print(f'[v2] Live remaining budget    : {REMAINING_REQUESTS} requests, {REMAINING_RESULTS} results')
print(f'[v2] Queries to fetch         : {len(to_fetch)}/{QUERIES_TO_FETCH}')
print(f'[v2] Size per query           : {SIZE_PER_QUERY}  ({PAGES_PER_QUERY} page(s)/query, {V2_MAX_PAGE_SIZE} max/page)')
print(f'[v2] Estimated cost this run  : {est_requests} requests, {est_results} results')
print(f'[v2] Evaluation k values      : {K_VALUES}')

In [ ]:
V2_COLUMNS            = ['domain', 'name', 'country', 'summary']
V2_RATE_LIMIT_RETRIES = 5
V2_RATE_LIMIT_WAIT    = 10   # seconds, multiplied by attempt number
V2_PAGE_SLEEP         = 1.5  # seconds between pages of the SAME query (separate from the
                              # 3.0s between-query sleep in the fetch loop below)


def _v2_call_page(query, size, search_after=None):
    """Single page of the v2 API. Pass search_after (the previous response's
    metadata.search_after cursor) to fetch the next page; omit/None for the
    first page. Per Istari's /api/goi-search docs: request field is top-level
    `search_after`, response field is `metadata.search_after` (null when
    there are no further pages).

    A 429 here can mean two different things and they need different handling:
      - Genuine MONTHLY quota exhaustion: X-RateLimit-Results-Remaining is at/near 0,
        or the body explicitly says so (e.g. "Monthly result quota exceeded"). Retrying
        never helps here, so this is fatal -- stop the whole run immediately.
      - A transient per-second/minute RATE LIMIT (e.g. body is just {"message":"Too Many
        Requests"} with plenty of results still remaining, as seen when this notebook
        fired 101 requests at 1/sec). This clears in seconds, so it's worth a short retry.
    """
    payload = {
        'describe': query,
        'keywords': {'must_all': [], 'must_any': [], 'must_not': []},
        'filters':  {'country': [], 'state': [], 'region': [],
                     'organization_type': [], 'organization_size': [], 'nace_code': []},
        'excludes': [],
        'columns':  V2_COLUMNS,
        'size':     size,
    }
    if search_after is not None:
        payload['search_after'] = search_after

    for attempt in range(V2_RATE_LIMIT_RETRIES):
        t0   = time.perf_counter()
        resp = requests.post(
            V2_BASE_URL,
            headers={
                'Accept': 'application/json',
                'x-api-key': ISTARI_API_KEY,
                'Content-Type': 'application/json',
            },
            json=payload,
            timeout=30,
        )
        ms = (time.perf_counter() - t0) * 1000
        remaining_hdr = resp.headers.get('X-RateLimit-Results-Remaining')
        remaining     = int(remaining_hdr) if remaining_hdr is not None else None

        if resp.status_code == 200:
            body        = resp.json()
            data        = body.get('data', [])
            next_cursor = body.get('metadata', {}).get('search_after')
            return data, ms, remaining, next_cursor

        if resp.status_code == 429:
            body_text = resp.text[:200]
            quota_exhausted = (remaining is not None and remaining <= 0) or 'quota' in body_text.lower()
            if quota_exhausted:
                raise SystemExit(
                    f'[v2] 429 — monthly quota genuinely exhausted on "{query}" '
                    f'(results remaining: {remaining}). Stopping — retrying won\'t help. '
                    f'Response: {body_text}'
                )
            wait = V2_RATE_LIMIT_WAIT * (attempt + 1)
            print(f'    [v2] 429 transient rate limit on "{query}" ({remaining} results still '
                  f'remaining — not a quota issue) — waiting {wait}s (attempt {attempt+1}/{V2_RATE_LIMIT_RETRIES})')
            time.sleep(wait)
            continue

        print(f'    [v2] ERROR {resp.status_code} on "{query}": {resp.text[:150]}')
        return None, ms, remaining, None  # None = genuine failure -- caller must NOT cache this

    print(f'    [v2] Gave up on "{query}" after {V2_RATE_LIMIT_RETRIES} rate-limit retries — skipping (not cached).')
    return None, 0, None, None


def v2_call(query, target_size, max_pages):
    """Fetches up to target_size UNIQUE-domain results for one query, paginating
    in pages of up to V2_MAX_PAGE_SIZE via the search_after cursor, capped at
    max_pages requests regardless of whether target_size was actually reached.

    Deduplicates by domain across pages: near-tied ranking scores can legitimately
    cause the same domain to reappear at a page boundary (observed live: 1-15
    duplicates out of 1000, i.e. <1.5%) -- this is normal cursor-pagination
    behaviour, NOT the catastrophic old `from`-offset bug (which duplicated ~75%
    of results, e.g. 50 unique out of 200). Deduplicating here means the cached
    output is always clean, so the staleness check in the fetch-loop cell can
    stay strict (within a small tolerance) without false-triggering on this
    harmless overlap.

    The max_pages cap matters because deduplication can leave a query a handful
    of results short of target_size (e.g. 985/1000) -- without the cap, the loop
    would keep requesting extra pages to chase those last few results, making the
    real request cost unpredictable and potentially exceeding the budget checked
    in the previous cell. Accepting a small, bounded shortfall is preferable to
    an open-ended request count.

    Also stops early if the API reports no further page (metadata.search_after
    is null) even if target_size hasn't been reached -- that means the corpus
    genuinely has fewer distinct matches than requested, not an error. Assigns a
    global rank (1..N) across all pages combined, over unique domains only."""
    all_data       = []
    seen_domains   = set()
    total_ms       = 0.0
    search_after   = None
    last_remaining = None
    pages_fetched  = 0

    while len(all_data) < target_size and pages_fetched < max_pages:
        page_size = min(V2_MAX_PAGE_SIZE, target_size - len(all_data))
        data, ms, remaining, next_cursor = _v2_call_page(query, page_size, search_after)
        pages_fetched += 1
        total_ms += ms
        if data is None:
            return None, total_ms, last_remaining  # propagate failure -- caller must NOT cache

        new_rows = [r for r in data if r['domain'] not in seen_domains]
        seen_domains.update(r['domain'] for r in new_rows)
        for rank, res in enumerate(new_rows, start=len(all_data) + 1):
            res['rank'] = rank
        all_data.extend(new_rows)
        last_remaining = remaining if remaining is not None else last_remaining

        if not data or next_cursor is None:
            break  # no further page available -- corpus has fewer distinct matches than target_size
        search_after = next_cursor
        if pages_fetched < max_pages:
            time.sleep(V2_PAGE_SLEEP)

    return all_data, total_ms, last_remaining


print('[v2] Functions defined ✅ (paginates via search_after up to '
      f'{V2_MAX_PAGE_SIZE}/page, deduplicates by domain across pages, capped at '
      'max_pages requests/query, fatal only on real quota exhaustion, short '
      'retry on transient rate limits, no cache poisoning)')

In [ ]:
# v2_results, v2_latencies, and to_fetch were already loaded/filtered in the
# previous cell (before the budget check), so this cell is just the fetch loop.

if to_fetch:
    print(f'[v2] Fetching {len(to_fetch)} queries at target size={SIZE_PER_QUERY} each '
          f'({PAGES_PER_QUERY} page(s)/query)...')
    print('-' * 60)
    total_start = time.time()

    for i, query in enumerate(to_fetch):
        data, ms, remaining = v2_call(query, target_size=SIZE_PER_QUERY, max_pages=PAGES_PER_QUERY)

        if data is None:
            print(f'[v2] Skipping "{query}" due to error above — NOT cached, will retry next run.')
            continue

        v2_results[query]   = data
        v2_latencies[query] = ms

        with open(v2_cache_path, 'wb') as f:
            pickle.dump((v2_results, v2_latencies), f)

        if (i + 1) % 10 == 0 or (i + 1) == len(to_fetch):
            elapsed     = time.time() - total_start
            remaining_t = (len(to_fetch) - i - 1) * elapsed / (i + 1) if (i + 1) > 0 else 0
            avg_ms      = sum(v2_latencies.values()) / len(v2_latencies)
            print(f'[v2] {i+1:3d}/{len(to_fetch)}  |  '
                  f'{len(data)} results  |  '
                  f'avg {avg_ms:.0f}ms/query  |  '
                  f'~{remaining_t:.0f}s remaining  |  '
                  f'quota results left: {remaining}')

        time.sleep(3.0)  # bumped from 1.0s — that pace tripped a transient 429 after 10 queries

    print('-' * 60)
    print(f'[v2] Done! {len(v2_results)} queries fetched.')
else:
    print('[v2] All queries already cached ✅')

with open(RESULT_DIR / 'v2_results.json', 'w') as f:
    json.dump(v2_results, f, indent=2, default=str)

sizes = [len(v2_results[q]) for q in v2_results]
print(f'[v2] Results per query: min={min(sizes)}  max={max(sizes)}  avg={sum(sizes)/len(sizes):.0f}')
print(f'[v2] Saved to {RESULT_DIR}/v2_results.json')

## 6 · Load Ground Truth

In [ ]:
print('[GT] Loading ground truth...')
with open('dataset/goi_search_results.json', 'r') as f:
    goi_data = json.load(f)

gt_scores_by_query = {}
gt_ranked_by_query = {}
for item in goi_data:
    q = item['query']
    sorted_results = sorted(item['results'], key=lambda x: x['rank'])
    gt_scores_by_query[q] = {r['domain']: float(r['similarity_score']) for r in sorted_results}
    gt_ranked_by_query[q] = [r['domain'] for r in sorted_results]

matched_queries = {}
for q in TEST_QUERIES:
    if q in gt_scores_by_query:
        matched_queries[q] = q
    else:
        for gt_q in gt_scores_by_query:
            if gt_q.lower().strip() == q.lower().strip():
                matched_queries[q] = gt_q
                break

print(f'[GT] Loaded {len(goi_data)} queries')
print(f'[GT] Matched {len(matched_queries)}/{len(TEST_QUERIES)} queries to ground truth')

## 7 · Evaluation

Metrics computed at each `k` in `K_VALUES` (set dynamically in Section 5 based on the live
quota check and real `search_after` pagination — shallower than the target `{10, 30, 50,
100, 200, 500, 1000}` only when the remaining budget doesn't support that depth across all
101 queries):
- **NDCG (graded)** — uses raw similarity scores as relevance weights
- **Precision@k** — fraction of top-k above threshold 0.75
- **Recall@k** — fraction of all relevant companies retrieved
- **Overlap@k** — how many domains appear in both retrieved and GT top-k

In [ ]:
def ndcg_graded_at_k(retrieved, gt_scores, k):
    dcg  = sum(gt_scores.get(d, 0.0) / np.log2(i + 2)
               for i, d in enumerate(retrieved[:k]))
    idcg = sum(s / np.log2(i + 2)
               for i, s in enumerate(sorted(gt_scores.values(), reverse=True)[:k]))
    return dcg / idcg if idcg > 0 else 0

def precision_at_k(retrieved, gt_scores, k, thresh=RELEVANCE_THRESHOLD):
    return sum(1 for d in retrieved[:k] if gt_scores.get(d, 0) >= thresh) / k if k else 0

def recall_at_k(retrieved, gt_scores, k, thresh=RELEVANCE_THRESHOLD):
    total = sum(1 for s in gt_scores.values() if s >= thresh)
    return sum(1 for d in retrieved[:k] if gt_scores.get(d, 0) >= thresh) / total if total else 0

def f1_at_k(retrieved, gt_scores, k, thresh=RELEVANCE_THRESHOLD):
    p = precision_at_k(retrieved, gt_scores, k, thresh)
    r = recall_at_k(retrieved, gt_scores, k, thresh)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def overlap_at_k(retrieved, gt_ranked, k):
    return len(set(retrieved[:k]) & set(gt_ranked[:k]))

print('[Eval] Metric functions defined ✅')

In [ ]:
METHODS = [
    ('Istari v1 BM25',     api_results['bm25']),
    ('Istari v2 Semantic', v2_results),
    ('Our MiniLM (98K)',   our_results),
]

print('[Eval] Computing metrics for all methods and queries...')
eval_rows = []

for method_name, results_dict in METHODS:
    for query in TEST_QUERIES:
        gt_q = matched_queries.get(query)
        if not gt_q:
            continue
        gt_scores = gt_scores_by_query[gt_q]
        gt_ranked = gt_ranked_by_query[gt_q]
        retrieved = [r['domain'] for r in
                     sorted(results_dict.get(query, []), key=lambda x: x.get('rank', 9999))]
        for k in K_VALUES:
            eval_rows.append({
                'method':    method_name,
                'query':     query,
                'k':         k,
                'ndcg':      round(ndcg_graded_at_k(retrieved, gt_scores, k), 4),
                'precision': round(precision_at_k(retrieved, gt_scores, k), 4),
                'recall':    round(recall_at_k(retrieved, gt_scores, k), 4),
                'f1':        round(f1_at_k(retrieved, gt_scores, k), 4),
                'overlap':   overlap_at_k(retrieved, gt_ranked, k),
            })

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_v2_comparison.csv', index=False)
print(f'[Eval] Done — {len(eval_df):,} rows saved to {RESULT_DIR}/evaluation_v2_comparison.csv')

## 8 · Results Table

In [ ]:
print('=' * 75)
print(f'EVALUATION vs GOI GROUND TRUTH  (threshold={RELEVANCE_THRESHOLD})')
print('=' * 75)
print(f'  {"Method":<24} {"k":>6} | {"NDCG":>7} | {"Prec":>7} | {"Recall":>7} | {"F1":>7} | {"Overlap":>8}')
print('  ' + '=' * 75)

for method_name, _ in METHODS:
    for k in K_VALUES:
        sub = eval_df[(eval_df['method'] == method_name) & (eval_df['k'] == k)]
        if len(sub) == 0:
            continue
        print(f'  {method_name:<24} {k:>6} | '
              f'{sub["ndcg"].mean():>7.3f} | '
              f'{sub["precision"].mean():>7.3f} | '
              f'{sub["recall"].mean():>7.3f} | '
              f'{sub["f1"].mean():>7.3f} | '
              f'{sub["overlap"].mean():>8.1f}')
    print('  ' + '-' * 75)

## 9 · Latency Summary

In [ ]:
avg_v1    = np.mean(list(api_latencies['bm25'].values())) if api_latencies['bm25'] else 0
avg_v2    = np.mean(list(v2_latencies.values()))          if v2_latencies          else 0
steady_keys = list(our_latencies.keys())[5:]
steady_ml   = np.mean([our_latencies[q] for q in steady_keys if q in our_latencies])

print('[Latency] ============================================================')
print(f'  Istari v1 BM25 (network)     : {avg_v1:.0f}ms avg  (2 × 500 per query)')
print(f'  Istari v2 Semantic (network) : {avg_v2:.0f}ms avg  (sum of real request time across '
      f'{PAGES_PER_QUERY} page(s)/query at size={SIZE_PER_QUERY}; excludes the {V2_PAGE_SLEEP}s '
      f'inter-page sleep, which is our own rate-limit safety margin, not API latency)')
print(f'  Our MiniLM (local A100)      : {steady_ml:.1f}ms steady-state')
print('[Latency] NOTE: this is latency for a full deep fetch (up to 1000 results), NOT directly')
print('[Latency] comparable to production\'s typical per-search latency -- see the benchmark below.')

latency_summary = pd.DataFrame([
    {'method': 'Istari v1 BM25',     'avg_ms': round(avg_v1, 1),    'note': '2 network calls × 500'},
    {'method': 'Istari v2 Semantic',  'avg_ms': round(avg_v2, 1),    'note': f'{PAGES_PER_QUERY} page(s) × {SIZE_PER_QUERY // PAGES_PER_QUERY} (approx), full 20M GOI'},
    {'method': 'Our MiniLM (98K)',    'avg_ms': round(steady_ml, 1), 'note': 'local A100 GPU'},
])
latency_summary.to_csv(RESULT_DIR / 'latency_summary.csv', index=False)
print(f'[Latency] Saved to {RESULT_DIR}/latency_summary.csv')

## 9b · Comparable Latency Benchmark (vs Production Baseline)

The measurement above times a full deep fetch (up to 1000 results via pagination), which is
not what a typical production search does. Manav's production latency baseline
(`latency-baseline.txt`) reports percentiles for the real search route:

| Scope | Requests | p50 | p75 | p95 | Avg | Max |
|---|---:|---:|---:|---:|---:|---:|
| Search route | 39,039 | 466 ms | 553 ms | 2,143 ms | 974 ms | 181,012 ms |

To compare like-for-like, this section times a small sample of **single-page, small-size**
v2 requests (`size=25`, no pagination) — the same scale as a typical UI search — and reports
the same percentile table, so the two are directly comparable instead of comparing a
deliberately heavy 1000-result/2-page fetch against typical production usage.

In [ ]:
LATENCY_BENCH_SIZE = 25   # typical single-page UI result count, not a deep fetch
LATENCY_BENCH_N    = 30   # sample of queries -- trivial quota cost (<=30 requests, <=750 results)

bench_cache_path = RESULT_DIR / 'latency_bench_cache.pkl'
if bench_cache_path.exists():
    with open(bench_cache_path, 'rb') as f:
        bench_latencies_ms = pickle.load(f)
    print(f'[LatencyBench] Loaded {len(bench_latencies_ms)} cached timings from a previous run')
else:
    bench_queries = TEST_QUERIES[:LATENCY_BENCH_N]
    bench_latencies_ms = []

    print(f'[LatencyBench] Timing {len(bench_queries)} single-page requests at size={LATENCY_BENCH_SIZE} '
          f'(comparable to production\'s typical search-route latency, not the 1000-result fetch above)...')

    for query in bench_queries:
        data, ms, remaining, _ = _v2_call_page(query, size=LATENCY_BENCH_SIZE)
        if data is not None:
            bench_latencies_ms.append(ms)
        time.sleep(1.0)

    with open(bench_cache_path, 'wb') as f:
        pickle.dump(bench_latencies_ms, f)
    print(f'[LatencyBench] Done — {len(bench_latencies_ms)}/{len(bench_queries)} timed successfully')

bench_arr = np.array(bench_latencies_ms)
p50, p75, p95 = np.percentile(bench_arr, [50, 75, 95])
avg_bench = bench_arr.mean()
max_bench = bench_arr.max()

print('[LatencyBench] ============================================================')
print(f'  v2 Semantic (size={LATENCY_BENCH_SIZE}, n={len(bench_arr)}) : '
      f'p50={p50:.0f}ms  p75={p75:.0f}ms  p95={p95:.0f}ms  avg={avg_bench:.0f}ms  max={max_bench:.0f}ms')
print(f'  Production search route (n=39,039, per Manav) : '
      f'p50=466ms  p75=553ms  p95=2143ms  avg=974ms  max=181012ms')
print('[LatencyBench] ============================================================')

latency_bench_df = pd.DataFrame([
    {'scope': f'v2 Semantic (this notebook, size={LATENCY_BENCH_SIZE}, single page)',
     'n': len(bench_arr), 'p50_ms': round(p50, 1), 'p75_ms': round(p75, 1),
     'p95_ms': round(p95, 1), 'avg_ms': round(avg_bench, 1), 'max_ms': round(max_bench, 1)},
    {'scope': 'Production search route (per Manav, latency-baseline.txt)',
     'n': 39039, 'p50_ms': 466, 'p75_ms': 553, 'p95_ms': 2143, 'avg_ms': 974, 'max_ms': 181012},
])
latency_bench_df.to_csv(RESULT_DIR / 'latency_benchmark_vs_production.csv', index=False)
print(f'[LatencyBench] Saved to {RESULT_DIR}/latency_benchmark_vs_production.csv')

## 10 · Key Findings

In [ ]:
print('[Findings] ============================================================')
print('[Findings] KEY COMPARISON: v2 Semantic vs Our MiniLM')
print('[Findings] ============================================================')

for k in K_VALUES:
    v2_sub = eval_df[(eval_df['method'] == 'Istari v2 Semantic') & (eval_df['k'] == k)]
    ml_sub = eval_df[(eval_df['method'] == 'Our MiniLM (98K)')   & (eval_df['k'] == k)]
    bm_sub = eval_df[(eval_df['method'] == 'Istari v1 BM25')     & (eval_df['k'] == k)]
    if len(v2_sub) == 0 or v2_sub['ndcg'].mean() == 0:
        continue
    v2_ndcg, ml_ndcg, bm_ndcg = v2_sub['ndcg'].mean(), ml_sub['ndcg'].mean(), bm_sub['ndcg'].mean()
    v2_rec,  ml_rec            = v2_sub['recall'].mean(), ml_sub['recall'].mean()
    delta = (ml_ndcg / v2_ndcg - 1) * 100 if v2_ndcg > 0 else float('inf')
    # Which one actually has the higher NDCG at this k -- do not assume it's always v2.
    v2_label = '  ← higher NDCG' if v2_ndcg >= ml_ndcg else ''
    ml_label = '  ← higher NDCG' if ml_ndcg > v2_ndcg else ''
    print(f'\n  k={k}:')
    print(f'    Istari v1 BM25      NDCG={bm_ndcg:.3f}')
    print(f'    Istari v2 Semantic  NDCG={v2_ndcg:.3f}  Recall={v2_rec:.3f}{v2_label}')
    print(f'    Our MiniLM (98K)    NDCG={ml_ndcg:.3f}  Recall={ml_rec:.3f}  ({delta:+.1f}% vs v2){ml_label}')

print('\n[Findings] ============================================================')
print('[Findings] RECALL GAP AT DEEPEST k')
print('[Findings] ============================================================')

deepest_k = max(K_VALUES)
v2_r = eval_df[(eval_df['method']=='Istari v2 Semantic') & (eval_df['k']==deepest_k)]['recall'].mean()
ml_r = eval_df[(eval_df['method']=='Our MiniLM (98K)')   & (eval_df['k']==deepest_k)]['recall'].mean()

if v2_r > 0 or ml_r > 0:
    gap = (v2_r - ml_r) * 100
    print(f'\n  Recall@{deepest_k} — v2 Semantic (live, 20M): {v2_r:.3f}   MiniLM (98K, local): {ml_r:.3f}')
    if gap > 0.5:
        print(f'  Gap: {gap:+.1f} pp — v2 recalls MORE than MiniLM at this depth.')
        print(f'  Consistent with a CORPUS COVERAGE gap: v2 searches the full 20M GOI, MiniLM only the 98K subsample.')
    elif gap < -0.5:
        print(f'  Gap: {gap:+.1f} pp — MiniLM recalls MORE than live v2 Semantic at this depth.')
        print(f'  CAVEAT: the ground truth (goi_search_results.json) is a FROZEN SNAPSHOT of production\'s')
        print(f'  semantic results from an earlier point in time, while this run queried the LIVE v2 API today.')
        print(f'  This result may reflect MiniLM agreeing more closely with that old snapshot than today\'s live')
        print(f'  v2 system does (i.e. v2 may have drifted/changed since the snapshot), rather than MiniLM')
        print(f'  strictly outperforming today\'s actual production quality. Confirm with Istari when the')
        print(f'  ground truth snapshot was captured before treating this as a definitive quality claim.')
    else:
        print(f'  Gap: {gap:+.1f} pp — v2 and MiniLM recall approximately equally at this depth.')

print('\n[Done] result/09b_api_v2/ — all files saved.')